In [1]:
!pip install --upgrade --force-reinstall --no-deps --no-cache-dir unsloth unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 249.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 315.5 MB/s  0:00:00
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.11.9
    Uninstalling unsloth_zoo-2026.11.9:
      Successfully uninstalled unsloth_zoo-2026.11.9
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.11.9
    Uninstalling unsloth-2026.11.9:
      Successfully uninstalled unsloth-2026.11.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [unsloth]


In [2]:
!pip install --upgrade "transformers<=5" trl  bitsandbytes

In [3]:
!pip install bitsandbytes

In [4]:
import pandas as pd
import torch
from unsloth import FastLanguageModel
import bitsandbytes

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/cleaned_data.csv")
print(data)

                                                     user  \
0       Explain tradeoffs between fiscal and monetary ...   
1       Explain the classical economic theory and its ...   
2       Explain the difference between fiscal and mone...   
3       Explain how central banks determine currency e...   
4       Explain how interest rates change with inflati...   
...                                                   ...   
423890  Can you explain the main differences between t...   
423891  Considering the text, what are the long-term f...   
423892  Can you explain the concept of a "negative fee...   
423893  How does Easterlin's book address the relation...   
423894  Can you explain the concept of the "undergroun...   

                                                assistant  
0       Fiscal and monetary policy are the two main to...  
1       The classical economic theory refers to the ec...  
2       Fiscal policy and monetary policy are the two ...  
3       1. Interest rates: 

In [6]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}")

CUDA available: True
GPU NVIDIA A100-SXM4-40GB


In [12]:
model_name = "unsloth/Qwen2.5-1.5B-Instruct"

max_token_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_token_seq_length,
    dtype=None,
    load_in_4bit=False
)

==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
from datasets import Dataset

def to_text(row):
    messages = [
        {"role": "user", "content": row["user"]},
        {"role": "assistant", "content": row["assistant"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,   # Qwen3: your data has no reasoning traces
        )
    }

dataset = Dataset.from_pandas(data)
dataset = dataset.map(to_text, remove_columns=dataset.column_names)
print(dataset[0]["text"])


Map:   0%|          | 0/423895 [00:00<?, ? examples/s]

<|im_start|>user
Explain tradeoffs between fiscal and monetary policy as tools in a nation's economic toolkit. Provide examples of past instances when each were utilized, the economic conditions that led to them being deployed, their intended effects, and an evaluation of their relative efficacy and consequences.<|im_end|>
<|im_start|>assistant
<think>

</think>

Fiscal and monetary policy are the two main tools that governments have to influence economic activity. They each have benefits and drawbacks.

Fiscal policy refers to government spending and taxation decisions. Examples of fiscal policy include:

• During the Great Recession, the U.S. government implemented a fiscal stimulus through the American Recovery and Reinvestment Act of 2009. This included increased spending on infrastructure, tax cuts, and expanded unemployment benefits. The intention was to boost aggregate demand and stimulate economic activity. Studies have found that the stimulus had a positive but modest impact o

In [14]:
#LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory - I use 64 because of the dataset size
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (general rule of thumb 2x rank) - scales the weight changes by alpha/r before adding to the freezed weight matrix
    lora_dropout=0,  # Supports any, but = 0 is optimized - can make higher if overfitting happens
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [15]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,
        dataset_num_proc=2,
        packing=True,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,   # effective batch = 8
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/423895 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/423895 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [16]:
# Train the model
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 92,963 | Num Epochs = 1 | Total steps = 11,621
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 73,859,072 of 1,617,573,376 (4.57% trained)


Step,Training Loss
25,1.615167
50,1.451015
75,1.278679
100,1.333271
125,1.240362
150,1.327519
175,1.210325
200,1.283531
225,1.321963
250,1.279774


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-11621/tokenizer_config.json.


In [17]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Explain the difference between inflation and deflation, with one historical example of each."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,      # match training format
    return_tensors="pt",
    return_dict=True,           # safer on newer transformers
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
)

# Decode only the newly generated tokens, not the prompt
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

<think>

</think>

Inflation refers to an increase in the general price level of goods and services in an economy over time. When the cost of living rises, it leads to higher prices for consumers. The opposite is deflation, which is when the general price level falls over time. 

Historical examples:

1) In 1920 during the Great Depression, the U.S. experienced high levels of inflation due to bank runs, commodity shortages, and reduced consumer spending. Prices rose rapidly as businesses tried to cover losses.

2) In Japan from 1957-1983, there was a prolonged period of deflation known as the "Lost Decade." The country saw low inflation rates and stagnant economic growth. This led to a decline in consumer demand and investment.

The key differences are that inflation represents rising costs while deflation indicates falling costs. Both can have negative impacts on economies but they differ in terms of their effects on purchasing power, interest rates, and business cycles.


In [18]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in gguf_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `gguf_model`: 100%|██████████| 1/1 [00:00<00:00, 2437.13it/s]


Successfully copied all 1 files from cache to `gguf_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.46s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12631-mix-acc0843 (app-b12631-mix-acc0843-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf/Qwen2.5-1.5B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed suc

{'save_directory': 'gguf_model',
 'gguf_directory': 'gguf_model_gguf',
 'gguf_files': ['gguf_model_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'gguf_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [24]:
import os, glob
from google.colab import files

def find_gguf():
    return sorted(glob.glob("/content/**/*.gguf", recursive=True), key=os.path.getmtime, reverse=True)

gguf_files = find_gguf()

# No GGUF yet: export it from the trained model
if not gguf_files:
    print("No .gguf found, exporting (this can take several minutes)...")
    model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")
    gguf_files = find_gguf()

if gguf_files:
    gguf_file = gguf_files[0]
    size_gb = os.path.getsize(gguf_file) / 1e9
    print(f"Found: {gguf_file} ({size_gb:.2f} GB)")
    files.download(gguf_file)
else:
    print("Export finished but no .gguf was produced. Contents of /content:")
    for root, _, fs in os.walk("/content"):
        if "sample_data" in root or "/." in root:
            continue
        for f in fs:
            print(os.path.join(root, f))

Found: /content/gguf_model_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf (0.99 GB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>